# Standardized semantic + lexical jailbreak classifier — issue #5
**One self-contained notebook, 12 Python cells.** Original CSVs stay unchanged; the only competition output is `submission.csv`, exactly `Id,TARGET`, preserving test IDs/order and finite [0,1] scores. TARGET >= 0.5 means jailbreak, otherwise benign. Prompt text is untrusted data, never executable instructions. Scores are model estimates, not security guarantees.

## Reproduce
Python 3.11. In a virtual environment install `numpy==2.3.4 pandas==2.3.3 scipy==1.16.3 scikit-learn==1.7.2 nbformat==5.10.4 nbclient==0.10.2 ipykernel==7.0.1 threadpoolctl==3.6.0 onnxruntime==1.23.2 tokenizers==0.22.1`, select that Jupyter kernel, then Run All from the repository root. Packages are not automatically installed. CPU-only; exact lexical kernels use quadratic memory, with conservative matrix-allocation guards (not a total RAM guarantee).

On a clean cache the notebook downloads approximately 23 MB of pinned MiniLM assets from public GitHub Git-blob API URLs. Exact blob IDs, sizes and SHA-256 hashes are verified before loading. Assets live under ignored `.cache/minilm/`, never in Git or the submission. Later runs are offline with dependencies/assets cached. No credentials, prompts or labels are sent to the download source. Verified local ONNX data is loaded without remote Python code or pickle.

## Research and pretrained provenance
Before editing, web search covered semantic/lexical fusion and nested calibration; web fetch retrieved [issue #5](https://github.com/johndesantis/modeltrain/issues/5); `gh issue view` confirmed its file restrictions. References: [calibration documentation](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibratedClassifierCV.html), [embedding-classifier evaluation](https://arxiv.org/pdf/1806.06259).

Frozen `all-MiniLM-L6-v2` assets are pinned to [AOSSIE-Org/MoveYourBody commit 53c128d](https://github.com/AOSSIE-Org/MoveYourBody/tree/53c128d1d8555d71d65fdd6bc035a27743dd8dd6/assets/models); its README identifies the [upstream sentence-transformers model](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2). The mirror application is GPL-3.0; review upstream model terms separately. No application code is copied or binary redistributed. **Check competition external-pretrained-model rules and applicable licenses before submission.** No extra labeled prompts are used.

## What changed and why
The user reports **0.97482 public ROC-AUC** for the previous semantic/lexical model. Its local calibrated grouped ROC-AUC was **0.964970**. The new requested public target is **0.98**, not a result claimed here.

A bounded comparison varied semantic RBF gamma and embedding preprocessing inside the same five development folds. Each learned mean/scale/projection was fitted on training rows only. Uncalibrated margin comparisons:

| Semantic variant | Semantic-only AUC | 70/30 lexical-semantic AUC |
|---|---:|---:|
| Raw embeddings, gamma .25 | 0.921335 | 0.961095 |
| Raw embeddings, gamma .5 | 0.931139 | 0.964641 |
| Raw embeddings, gamma 1 (previous) | 0.938587 | 0.965822 |
| Raw embeddings, gamma 2 | 0.939225 | 0.964935 |
| Raw embeddings, gamma 4 | 0.908314 | 0.961262 |
| Training-centered then normalized | 0.940851 | 0.966000 |
| **Training-standardized then normalized (selected)** | **0.947568** | **0.967647** |
| Half-whitened training covariance then normalized | 0.930429 | 0.964083 |

The standardized 55/45 margin blend scored 0.967683, only 0.000036 above 70/30. The established 70/30 weight was retained to avoid chasing this tiny difference. Gamma1 and C10 are unchanged. These are measured development experiments, not rerun on every execution. The selected complete model is freshly calibrated/evaluated below.

Development folds and public feedback have been reused across many revisions; selection optimism remains. No independent untouched holdout is claimed. The seed-2026 split is a sensitivity check only and will not drive further tuning. Conditional bootstrap intervals exclude training/selection uncertainty. **The 0.98 public target remains unverified, and no leaderboard upload is performed.**

## Model and leakage boundaries
70% lexical/latent polynomial-SVM margin + 30% semantic RBF-SVM margin, followed by grouped sigmoid calibration. The lexical branch uses word 1–3 grams and char_wb 2–6 grams, min_df1, 180k caps, sublinear TF, equal L2 view weighting; lexical/256-dimensional latent margins are blended 80/20.

The frozen encoder tokenizes each prompt independently, covers all tokens in non-overlapping 254-token chunks plus CLS/SEP, mean-pools/normalizes each chunk, then token-count-averages and normalizes again. Every inference call contains one chunk, avoiding batch-dependent quantization. No tail is silently discarded. Row-local embedding memoization is label-free and learns no cross-row statistics. Precomputing train/test embeddings fits nothing on test data.

**New train-fitted semantic preprocessing** is `StandardScaler -> Normalizer -> SVC(C10, RBF gamma1)`. It is fitted only when the classifier fits: separately in each calibration training fold and outer training fold, then on all training rows for the final classifier. Test embeddings never contribute means/variances. Three group-disjoint inner folds calibrate the blended margin, then both branches refit on all available training rows. Five grouped outer folds evaluate the full predictor. IDs, source metadata and sample targets are excluded from features.

## Safeguards
Strict CSV structure/schema/labels, canonical Unicode IDs, normalized duplicate/conflict/overlap checks, invalid controls/Unicode rejection, verified byte-snapshot loading, unchanged source hashes, hardlink/symlink output protection and dense-kernel guards remain. Label-free cosine groups at 0.80 stay together in outer/inner folds; mixed-label groups and test near-matches are audit-only. Lexical grouping cannot guarantee semantic/source independence.

New resource limits reject prompts over **100,000 characters** or **32,768 WordPiece tokens** rather than truncating them. Embedding normalization fails closed on wrong dimensions, NaN/Inf or zero vectors, and checks hidden-state finiteness. Inputs within budget retain all chunks. Tests verify exact training-only semantic means/variances/sample counts, held-out-statistic isolation and immutable scaler state during inference, plus all new budget/encoder error paths.

Pinned asset hashes/sizes/input-output schemas/special token IDs, read-only embeddings, row order and long-input tail sensitivity are tested. Complete binary probabilities are validated for shape, finiteness, bounds and sum-to-one. The exporter verifies IDs/order/scores and input hashes, writes through an exclusive temp stream, flushes/fsyncs, validates CSV read-back and atomically replaces the output. Fault injection checks write/fsync/replace failures, changed inputs and corrupted read-back: the previous file must survive and temp files must be cleaned. No helper scripts, extra notebooks or report sidecars are created.

Passing these checks verifies execution and tested software behavior, **not 100% predictive accuracy, universal input coverage or a guaranteed leaderboard score**. All documentation, audit reports, outputs and checks remain in this notebook.

## Verified outcome
Fresh-kernel execution completed without cell errors: **136 checks passed**, 1,000 scored test rows and unchanged original CSV hashes. Final calibrated grouped development ROC-AUC is **0.967033**, versus **0.964970** previously, with unchanged input hashes, encoder assets, group assignments/seed and outer-fold configuration. Conditional group-bootstrap 95% interval: **0.957401–0.974978**. Runtime was approximately **7.9 minutes** with cached assets. This is a new submission; **0.98 public ROC-AUC remains unverified**.


In [1]:
from pathlib import Path
import csv, hashlib, json, platform, re, time, unicodedata, warnings
import numpy as np
import pandas as pd
import scipy, sklearn
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, brier_score_loss
from sklearn.neighbors import NearestNeighbors
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits
warnings.filterwarnings('error', category=ConvergenceWarning)
SEED = 42
FOLDS = 5
NEAR_DUPLICATE_COSINE = 0.80
ROOT = Path.cwd()
START = time.monotonic()
# Avoid oversubscribed BLAS on CPU environments.
thread_limit = threadpool_limits(limits=2)
print('Python', platform.python_version(), '| sklearn', sklearn.__version__)
from io import StringIO
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import Normalizer
from sklearn.svm import SVC
INPUT_HASHES = {f: hashlib.sha256((ROOT/f).read_bytes()).hexdigest()
                for f in ['train.csv','test.csv','sample_submission.csv']}

import tempfile, os
from sklearn.metrics import average_precision_score

MAX_PROMPT_CHARACTERS = 100_000
MAX_SEMANTIC_TOKENS = 32_768


Python 3.11.2 | sklearn 1.7.2


## 1. Strict CSV validation (IDs are never predictive features)

In [2]:
def require(condition, message):
    if not condition:
        raise ValueError(message)

def normalize(text):
    text = unicodedata.normalize('NFKC', text).casefold()
    text = ''.join(c for c in text if unicodedata.category(c) != 'Cf')
    return ' '.join(text.split())

def validate_frame(frame, columns):
    require(list(frame.columns) == columns, f'Expected columns {columns}')
    require(len(frame) > 0, 'Empty dataset')
    require(not frame.isna().any().any(), 'Missing values')
    require(all(frame[col].map(lambda value: isinstance(value, str)).all() for col in columns),
            'CSV input fields must be strings')
    require(frame['Id'].str.strip().ne('').all(), 'Blank IDs')
    require(frame['Id'].is_unique, 'Duplicate IDs')
    require(not frame['Id'].map(lambda s: any(0xD800 <= ord(ch) <= 0xDFFF for ch in s)).any(),
            'Unpaired Unicode surrogate in ID')
    canonical_ids = frame['Id'].map(lambda s: unicodedata.normalize('NFKC', s))
    require(canonical_ids.is_unique, 'Unicode-equivalent duplicate IDs')
    require(not frame['Id'].map(lambda s: any(unicodedata.category(ch) == 'Cf' for ch in s)).any(),
            'Invisible formatting in IDs')
    require(frame['Id'].eq(frame['Id'].str.strip()).all(), 'Whitespace in IDs')
    require(not frame['Id'].str.contains(r'[\x00-\x1f\x7f]', regex=True).any(), 'Control character in ID')
    if 'text' in columns:
        require(frame.text.str.len().le(MAX_PROMPT_CHARACTERS).all(), 'Prompt character budget exceeded; no truncation performed')
        require(frame.text.map(normalize).ne('').all(), 'Empty normalized prompts')
        require(not frame.text.str.contains('\x00', regex=False).any(), 'NUL in prompt')
        require(not frame.text.str.contains(r'[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]', regex=True).any(),
                'Nontext control character in prompt')
        require(not frame.text.map(lambda s: any(0xD800 <= ord(ch) <= 0xDFFF for ch in s)).any(),
                'Unpaired Unicode surrogate in prompt')
    if 'label' in columns:
        require(set(frame.label) == {'benign', 'jailbreak'}, 'Expected both binary labels')
    return frame

def parse_csv(stream, columns):
    reader = csv.reader(stream, strict=True)
    try:
        require(next(reader, None) == columns, 'Invalid CSV header')
        rows = list(reader)
    except csv.Error as exc:
        raise ValueError('Malformed CSV quoting') from exc
    require(all(len(row) == len(columns) for row in rows), 'Malformed CSV row width')
    return validate_frame(pd.DataFrame(rows, columns=columns, dtype=str), columns)

def parse_verified_snapshot(data, expected_sha256, columns):
    require(hashlib.sha256(data).hexdigest() == expected_sha256, 'CSV snapshot hash mismatch')
    # Parse precisely the bytes that were verified, not a second read of a mutable path.
    return parse_csv(StringIO(data.decode('utf-8'), newline=''), columns)

def load_csv(name, columns):
    return parse_verified_snapshot((ROOT/name).read_bytes(), INPUT_HASHES[name], columns)

def validate_relations(train, test, sample):
    require(set(train.Id.map(lambda s: unicodedata.normalize('NFKC', s))).isdisjoint(
                test.Id.map(lambda s: unicodedata.normalize('NFKC', s))), 'Train/test canonical ID overlap')
    require(sample.Id.tolist() == test.Id.tolist(), 'Sample/test ID order mismatch')
    require(set(train.text.map(normalize)).isdisjoint(test.text.map(normalize)),
            'Normalized train/test text overlap')
    require(train.assign(key=train.text.map(normalize)).groupby('key').label.nunique().max() == 1,
            'Conflicting labels for identical normalized prompts')
    sample_scores = pd.to_numeric(sample.TARGET, errors='coerce').to_numpy()
    require(np.isfinite(sample_scores).all() and ((sample_scores >= 0) & (sample_scores <= 1)).all(),
            'Invalid sample targets')

train = load_csv('train.csv', ['Id', 'text', 'label'])
test = load_csv('test.csv', ['Id', 'text'])
sample = load_csv('sample_submission.csv', ['Id', 'TARGET'])
validate_relations(train, test, sample)
train['normalized'] = train.text.map(normalize)
test['normalized'] = test.text.map(normalize)
require(set(train.normalized).isdisjoint(test.normalized), 'Normalized train/test text overlap')
conflicts = train.groupby('normalized').label.nunique()
require(conflicts.max() == 1, 'Conflicting labels for identical normalized prompts')
duplicate_rows = int(train.normalized.duplicated().sum())
train = train.drop_duplicates('normalized').reset_index(drop=True)
texts = train.normalized.to_numpy()
y = train.label.map({'benign': 0, 'jailbreak': 1}).to_numpy()
print('Training:', len(train), '| test:', len(test), '| removed duplicates:', duplicate_rows)
print(train.label.value_counts().to_dict())

Training: 4000 | test: 1000 | removed duplicates: 0
{'benign': 2401, 'jailbreak': 1599}


## 2. Conservative near-duplicate grouping
Unsupervised hashing is used **only** for split grouping, never as predictive features. Connected components keep transitive template families in one fold. This is a lexical safeguard, not a semantic/source-independence guarantee.

In [3]:
from sklearn.feature_extraction.text import HashingVectorizer
# No learned vocabulary or IDF, no test prompts and no labels used for grouping.
group_vectorizer = HashingVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                                    n_features=2**18, alternate_sign=False, norm='l2')
group_matrix = group_vectorizer.transform(texts)
neighbor_index = NearestNeighbors(metric='cosine', algorithm='brute', n_jobs=2).fit(group_matrix)
graph = neighbor_index.radius_neighbors_graph(group_matrix,
    radius=1 - NEAR_DUPLICATE_COSINE + 1e-12, mode='connectivity')
_, groups = connected_components(graph, directed=False)
splits = list(StratifiedGroupKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
              .split(texts, y, groups))
for tr, va in splits:
    require(set(groups[tr]).isdisjoint(groups[va]), 'Group leakage')
    require(len(np.unique(y[tr])) == len(np.unique(y[va])) == 2, 'Single-class fold')
print('Groups:', len(np.unique(groups)), '| largest:', int(np.bincount(groups).max()))
print('Validation fold sizes:', [len(va) for _, va in splits])
# Audit only; test similarities never determine model weights or predictions.
nearest_distance, _ = neighbor_index.kneighbors(
    group_vectorizer.transform(test.normalized), n_neighbors=1)
near_test_count = int((nearest_distance[:, 0] <= 1 - NEAR_DUPLICATE_COSINE).sum())
print('Test prompts near a training prompt (audit only):', near_test_count)

# Flag mixed-label near-duplicate families for human review; never relabel or drop them.
mixed_groups = int((pd.DataFrame({'group':groups, 'label':y}).groupby('group').label.nunique() > 1).sum())
print('Mixed-label lexical groups (retained, kept within folds):', mixed_groups)
print('Prompt character lengths, train/test medians:',
      float(train.text.str.len().median()), float(test.text.str.len().median()))


Groups: 3855 | largest: 85
Validation fold sizes: [729, 803, 793, 791, 884]


Test prompts near a training prompt (audit only): 43
Mixed-label lexical groups (retained, kept within folds): 0
Prompt character lengths, train/test medians: 616.0 606.5


## 3. Lexical branch and frozen semantic features
The established lexical branch retains word/character TF-IDF and an 80/20 lexical/latent polynomial SVM. The following cells load checksum-pinned frozen encoder assets and construct independent semantic features. The hybrid estimator then blends lexical and semantic margins before group-disjoint calibration, followed by a full-data refit of both classifiers. IDs are never predictive features. Complete binary distributions are validated before selecting the positive-class score.

## Frozen pretrained encoder — verified assets

In [4]:
# Frozen encoder assets: pin immutable Git blobs and verify exact bytes before loading.
import urllib.request
import onnxruntime as ort
from tokenizers import BertWordPieceTokenizer
import tokenizers
ort.disable_telemetry_events()
ASSET_SOURCE_COMMIT = '53c128d1d8555d71d65fdd6bc035a27743dd8dd6'
ASSET_REPOSITORY = 'AOSSIE-Org/MoveYourBody'
ASSETS = {
    'model.onnx': ('3ff3bf5e385e3b2189b18aff47fa0f605060a995',
                   '321dc4b9e00da5e056106d884e02094b678583bee558796eb66f66e4f26ede83', 22843709),
    'vocab.txt': ('fb140275c155a9c7c5a3b3e0e77a9e839594a938',
                  '07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3', 231508),
}
ASSET_DIR = ROOT/'.cache'/'minilm'
ASSET_DIR.mkdir(parents=True, exist_ok=True)

def verify_asset_payload(payload, digest, expected_size):
    require(len(payload) == expected_size, 'Encoder asset size mismatch')
    require(hashlib.sha256(payload).hexdigest() == digest, 'Encoder asset SHA-256 mismatch')
    return payload

def encoder_asset(name):
    blob, digest, expected_size = ASSETS[name]
    path = ASSET_DIR/name
    require(not path.is_symlink(), 'Refusing symlinked encoder asset')
    if path.exists():
        verify_asset_payload(path.read_bytes(), digest, expected_size)
        return path
    url = f'https://api.github.com/repos/{ASSET_REPOSITORY}/git/blobs/{blob}'
    request = urllib.request.Request(url, headers={
        'Accept':'application/vnd.github.raw+json', 'User-Agent':'modeltrain-notebook'})
    # No credentials, prompt text, or labels are sent in this public asset request.
    with urllib.request.urlopen(request, timeout=120) as response:
        require(response.geturl().startswith('https://api.github.com/'), 'Unexpected asset redirect')
        payload = response.read(expected_size+1)
    verify_asset_payload(payload, digest, expected_size)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=ASSET_DIR, prefix='.asset-', delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(payload)
            stream.flush()
            os.fsync(stream.fileno())
        temporary.replace(path)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)
    return path

model_path = encoder_asset('model.onnx')
vocab_path = encoder_asset('vocab.txt')
options = ort.SessionOptions()
options.intra_op_num_threads = 2
options.inter_op_num_threads = 1
encoder = ort.InferenceSession(str(model_path), sess_options=options, providers=['CPUExecutionProvider'])
require([x.name for x in encoder.get_inputs()] == ['input_ids','attention_mask','token_type_ids'],
        'Unexpected encoder input schema')
wordpiece = BertWordPieceTokenizer(str(vocab_path), lowercase=True)
require([wordpiece.token_to_id(x) for x in ['[PAD]','[CLS]','[SEP]']] == [0,101,102],
        'Unexpected tokenizer special IDs')
print('Frozen MiniLM loaded from verified GitHub assets; no encoder training.')
print('Model SHA-256:', ASSETS['model.onnx'][1])


Frozen MiniLM loaded from verified GitHub assets; no encoder training.
Model SHA-256: 321dc4b9e00da5e056106d884e02094b678583bee558796eb66f66e4f26ede83


## Independent full-prompt semantic features

In [5]:
# Each prompt is encoded independently. Chunk all tokens; never discard the tail.
SEMANTIC_DIMENSION = 384
SEMANTIC_WEIGHT = 0.3
_embedding_cache = {}
_chunk_counts = {}

def check_semantic_token_budget(token_count):
    require(isinstance(token_count, int) and not isinstance(token_count, bool) and token_count >= 0,
            'Invalid semantic token count')
    require(token_count <= MAX_SEMANTIC_TOKENS, 'Semantic token budget exceeded; no truncation performed')

def normalize_encoder_vector(vector):
    vector = np.asarray(vector,dtype=np.float64)
    require(vector.shape == (SEMANTIC_DIMENSION,), 'Invalid encoder vector dimension')
    require(np.isfinite(vector).all(), 'Nonfinite encoder vector')
    norm = float(np.linalg.norm(vector))
    require(np.isfinite(norm) and norm > 1e-12, 'Degenerate encoder vector')
    return vector/norm

def frozen_embedding(prompt):
    require(isinstance(prompt,str) and len(prompt) <= MAX_PROMPT_CHARACTERS, 'Invalid or over-budget semantic prompt')
    key = normalize(prompt)
    require(bool(key), 'Empty semantic prompt')
    if key in _embedding_cache:
        return _embedding_cache[key]
    ids = wordpiece.encode(key, add_special_tokens=False).ids
    check_semantic_token_budget(len(ids))
    chunks = [ids[i:i+254] for i in range(0,len(ids),254)] or [[]]
    vectors, weights = [], []
    for chunk in chunks:
        token_ids = np.asarray([[101]+chunk+[102]], dtype=np.int64)
        attention = np.ones_like(token_ids)
        # One chunk per inference call prevents batch-dependent quantization effects.
        hidden = encoder.run(None, {'input_ids':token_ids, 'attention_mask':attention,
                                    'token_type_ids':np.zeros_like(token_ids)})[0]
        require(hidden.shape == (1,len(chunk)+2,SEMANTIC_DIMENSION), 'Unexpected encoder output shape')
        require(np.isfinite(hidden).all(), 'Nonfinite encoder hidden state')
        vector = normalize_encoder_vector(hidden[0].mean(axis=0))
        vectors.append(vector)
        weights.append(max(1,len(chunk)))
    require(sum(weights) == max(1,len(ids)), 'Encoder chunk coverage mismatch')
    result = normalize_encoder_vector(np.average(vectors,axis=0,weights=weights))
    require(result.shape == (SEMANTIC_DIMENSION,) and np.isfinite(result).all(), 'Invalid semantic embedding')
    result.setflags(write=False)
    _embedding_cache[key] = result
    _chunk_counts[key] = len(chunks)
    return result

def semantic_features(prompts):
    return np.vstack([frozen_embedding(prompt) for prompt in prompts])

# A frozen, row-local encoder learns no vocabulary/statistics from these rows.
# Precomputation uses no labels; CV classifiers/calibrators still fit only training folds.
for i,prompt in enumerate(list(texts)+test.normalized.tolist(),start=1):
    frozen_embedding(prompt)
    if i%1000 == 0:
        print('Encoded prompts:',i,flush=True)
print('Semantic width:',SEMANTIC_DIMENSION,'| maximum token chunks:',max(_chunk_counts.values()))


Encoded prompts: 1000


Encoded prompts: 2000


Encoded prompts: 3000


Encoded prompts: 4000


Encoded prompts: 5000


Semantic width: 384 | maximum token chunks: 11


In [6]:
def vectorizer(view):
    common = dict(sublinear_tf=True, min_df=1, max_df=1.0, dtype=np.float64)
    if view == 'word':
        return TfidfVectorizer(ngram_range=(1, 3), token_pattern=r'(?u)\b\w+\b',
                               max_features=180000, **common)
    return TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 6),
                           max_features=180000, **common)

def polynomial_cosine(X, Y):
    return (1.0 + (X @ Y.T).toarray()) ** 2

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.preprocessing import normalize as l2_normalize
from scipy.linalg import eigh

MAX_KERNEL_WORKING_BYTES = 2 * 1024**3

def check_kernel_budget(training_rows, prediction_rows=0, maximum=MAX_KERNEL_WORKING_BYTES):
    # Conservative dense-matrix working-set estimate, not a total process-RAM guarantee.
    require(training_rows >= 2 and prediction_rows >= 0, 'Invalid kernel dimensions')
    estimate = 8 * (4 * training_rows**2 + 3 * training_rows * prediction_rows)
    require(estimate <= maximum, f'Estimated dense-kernel working set {estimate:,} exceeds budget {maximum:,}')
    return estimate

class LexicalLatentSVM(ClassifierMixin, BaseEstimator):
    """Two training-only polynomial SVMs; fixed 80/20 margin blend.

    Eigenvectors of the training cosine Gram matrix implement exact truncated
    LSA. Validation/test projections use the training basis, never a joint SVD.
    """
    def __init__(self, C=10.0, latent_components=256, latent_weight=0.2, random_state=42):
        self.C = C
        self.latent_components = latent_components
        self.latent_weight = latent_weight
        self.random_state = random_state

    def fit(self, X, y):
        require(X.shape[0] == len(y), 'Feature/label length mismatch')
        require(np.isfinite(X.data).all(), 'Nonfinite training features')
        require(0 <= self.latent_weight <= 1, 'Invalid latent blend weight')
        require(isinstance(self.latent_components, int) and self.latent_components >= 1,
                'Invalid latent rank')
        check_kernel_budget(X.shape[0])
        self.training_features_ = X.copy().tocsr()
        self.n_features_in_ = X.shape[1]
        self.classes_ = np.unique(y)
        require(self.classes_.tolist() == [0, 1], 'Expected both model classes')
        cosine = (X @ X.T).toarray()
        rank = min(self.latent_components, X.shape[0] - 1)
        values, vectors = eigh(cosine, subset_by_index=[X.shape[0]-rank, X.shape[0]-1],
                               check_finite=True)
        values = np.maximum(values, 1e-10)
        self.projection_ = vectors / np.sqrt(values)
        self.latent_training_ = l2_normalize(vectors * np.sqrt(values))
        self.lexical_ = SVC(C=self.C, kernel='precomputed', random_state=self.random_state)
        self.latent_ = SVC(C=self.C, kernel='precomputed', random_state=self.random_state)
        self.lexical_.fit((1.0 + cosine)**2, y)
        self.latent_.fit((1.0 + self.latent_training_ @ self.latent_training_.T)**2, y)
        return self

    def decision_function(self, X):
        check_is_fitted(self, ['lexical_', 'latent_', 'projection_'])
        require(X.shape[1] == self.n_features_in_, 'Inference feature dimension mismatch')
        require(np.isfinite(X.data).all(), 'Nonfinite inference features')
        check_kernel_budget(self.training_features_.shape[0], X.shape[0])
        cosine = (X @ self.training_features_.T).toarray()
        latent = l2_normalize(cosine @ self.projection_)
        lexical_margin = self.lexical_.decision_function((1.0 + cosine)**2)
        latent_margin = self.latent_.decision_function((1.0 + latent @ self.latent_training_.T)**2)
        return (1.0-self.latent_weight)*lexical_margin + self.latent_weight*latent_margin

    def predict(self, X):
        return self.classes_[(self.decision_function(X) >= 0).astype(int)]

def make_pipeline():
    return Pipeline([
        ('features', FeatureUnion([('word',vectorizer('word')), ('char',vectorizer('char'))])),
        ('normalize', Normalizer()),
        ('svm', LexicalLatentSVM(C=10, latent_components=256, latent_weight=.2, random_state=SEED))])

from sklearn.preprocessing import StandardScaler

def make_semantic_pipeline(random_state=SEED):
    # Statistics are fitted by .fit() only on the classifier's current training fold.
    return Pipeline([
        ('scale', StandardScaler()),
        ('normalize', Normalizer()),
        ('svm', SVC(C=10,kernel='rbf',gamma=1,cache_size=512,random_state=random_state)),
    ])

class SemanticLexicalClassifier(ClassifierMixin, BaseEstimator):
    """70% lexical/latent margin + 30% frozen-embedding RBF-SVM margin."""
    def __init__(self, semantic_weight=0.3, random_state=42):
        self.semantic_weight = semantic_weight
        self.random_state = random_state

    def fit(self, X, y):
        require(0 <= self.semantic_weight <= 1, 'Invalid semantic blend weight')
        self.classes_ = np.unique(y)
        require(self.classes_.tolist() == [0,1], 'Expected both hybrid model classes')
        self.lexical_pipeline_ = make_pipeline()
        self.lexical_pipeline_.fit(X,y)
        self.semantic_model_ = make_semantic_pipeline(random_state=self.random_state)
        features = semantic_features(X)
        self.semantic_model_.fit(features,y)
        require(self.semantic_model_.classes_.tolist() == self.classes_.tolist(), 'Semantic class order mismatch')
        self.semantic_training_rows_ = len(X)
        return self

    def decision_function(self, X):
        check_is_fitted(self, ['lexical_pipeline_','semantic_model_'])
        lexical = self.lexical_pipeline_.decision_function(X)
        semantic = self.semantic_model_.decision_function(semantic_features(X))
        return (1-self.semantic_weight)*lexical + self.semantic_weight*semantic

    def predict(self, X):
        return self.classes_[(self.decision_function(X) >= 0).astype(int)]

def validate_probability_matrix(probabilities, row_count):
    probabilities = np.asarray(probabilities)
    require(probabilities.shape == (row_count, 2), 'Invalid binary probability shape')
    require(np.isfinite(probabilities).all(), 'Nonfinite binary probability')
    require(((probabilities >= 0) & (probabilities <= 1)).all(), 'Binary probability outside [0,1]')
    require(np.allclose(probabilities.sum(axis=1), 1, rtol=0, atol=1e-10),
            'Binary probabilities do not sum to one')
    return probabilities[:, 1]

def fit_model(train_text, labels, prediction_text, train_groups):
    inner = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
                 .split(train_text, labels, train_groups))
    for tr, va in inner:
        require(set(train_groups[tr]).isdisjoint(train_groups[va]), 'Inner group leakage')
        require(len(np.unique(labels[tr])) == len(np.unique(labels[va])) == 2,
                'Inner fold lacks a class')
    model = CalibratedClassifierCV(SemanticLexicalClassifier(semantic_weight=SEMANTIC_WEIGHT, random_state=SEED), method='sigmoid', cv=inner,
                                  ensemble=False, n_jobs=1)
    model.fit(train_text, labels)
    require(model.classes_.tolist() == [0, 1], 'Incorrect class direction')
    return validate_probability_matrix(model.predict_proba(prediction_text), len(prediction_text)), model


## 4. Five-fold out-of-fold evaluation (no test feedback)

In [7]:
oof_score = np.full(len(train), np.nan)
fold_assignment = np.full(len(train), -1)
fold_metrics = []
for fold, (tr, va) in enumerate(splits):
    oof_score[va], _ = fit_model(texts[tr], y[tr], texts[va], groups[tr])
    fold_assignment[va] = fold
    auc = roc_auc_score(y[va], oof_score[va])
    fold_metrics.append(float(auc))
    print(f'Fold {fold+1}: ROC-AUC={auc:.6f}', flush=True)
require(np.isfinite(oof_score).all(), 'Incomplete OOF predictions')
oof = oof_score[:, None]
metrics = {'semantic_lexical_svm': float(roc_auc_score(y, oof_score))}
print('Grouped OOF ROC-AUC:', metrics)
print('Accuracy:', accuracy_score(y, oof_score >= .5),
      '| F1:', f1_score(y, oof_score >= .5), '| Brier:', brier_score_loss(y, oof_score))
print('Public ROC-AUC: unknown; 0.98 target remains unverified')
print('Average precision:', average_precision_score(y, oof_score))

# Predeclared seed-shift sensitivity check, not an independent untouched holdout.
# Do not change model settings based on this result; all rows were used during development.
sensitivity_split = next(StratifiedGroupKFold(5, shuffle=True, random_state=2026).split(texts,y,groups))
sensitivity_tr, sensitivity_va = sensitivity_split
require(set(groups[sensitivity_tr]).isdisjoint(groups[sensitivity_va]), 'Sensitivity group leakage')
sensitivity_score, _ = fit_model(texts[sensitivity_tr], y[sensitivity_tr],
                               texts[sensitivity_va], groups[sensitivity_tr])
sensitivity_auc = float(roc_auc_score(y[sensitivity_va], sensitivity_score))
print('Seed-2026 grouped split sensitivity ROC-AUC:', sensitivity_auc,
      '| validation rows:', len(sensitivity_va), flush=True)


Fold 1: ROC-AUC=0.967203


Fold 2: ROC-AUC=0.976976


Fold 3: ROC-AUC=0.969327


Fold 4: ROC-AUC=0.964893


Fold 5: ROC-AUC=0.964311


Grouped OOF ROC-AUC: {'semantic_lexical_svm': 0.9670327065619678}
Accuracy: 0.90225 | F1: 0.8737487891507911 | Brier: 0.07148229691599524
Public ROC-AUC: unknown; 0.98 target remains unverified
Average precision: 0.9540154186669275


Seed-2026 grouped split sensitivity ROC-AUC: 0.9768166089965398 | validation rows: 739


## 5. Group bootstrap uncertainty
Resampling whole groups accounts for lexical dependence, conditional on these fitted OOF predictions. It does not include model-selection/training uncertainty and is not a public-score interval.

In [8]:
rng = np.random.default_rng(SEED)
group_rows = [np.flatnonzero(groups == g) for g in np.unique(groups)]
bootstrap_auc = []
for _ in range(500):
    idx = np.concatenate([group_rows[g] for g in rng.integers(0, len(group_rows), len(group_rows))])
    if len(np.unique(y[idx])) == 2:
        bootstrap_auc.append(roc_auc_score(y[idx], oof_score[idx]))
auc_interval = np.quantile(bootstrap_auc, [.025, .975]).tolist()
print('Group-bootstrap 95% interval:', auc_interval)

Group-bootstrap 95% interval: [0.9574013516945595, 0.9749777990167104]


## 6. Refit on all labeled rows and score untouched test prompts

In [9]:
scores, final_model = fit_model(texts, y, test.normalized.to_numpy(), groups)
require(np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all(), 'Invalid probabilities')
print('Test score range:', float(scores.min()), float(scores.max()))
print('Threshold counts:', pd.Series(np.where(scores >= .5, 'jailbreak', 'benign')).value_counts().to_dict())

Test score range: 5.3464520625876146e-05 0.9999950712369745
Threshold counts: {'benign': 605, 'jailbreak': 395}


## 7. Fitted word-ID inspection
The unigram entries of the final word vocabulary supply stable IDs within this fitted model. Repeated words share IDs; unknown words map to -1. IDs are feature indices, not prompt IDs, and do not encode risk on their own.

In [10]:
final_hybrid = final_model.calibrated_classifiers_[0].estimator
final_pipeline = final_hybrid.lexical_pipeline_
word_vect = dict(final_pipeline.named_steps['features'].transformer_list)['word']
analyzer = word_vect.build_tokenizer()
def word_ids(prompt):
    return [(word, word_vect.vocabulary_.get(word, -1))
            for word in analyzer(normalize(prompt))]
print('Vocabulary entries (word 1–3 grams):', len(word_vect.vocabulary_))
print(word_ids(train.text.iloc[0])[:25])

Vocabulary entries (word 1–3 grams): 180000
[('you', np.int64(178024)), ('are', np.int64(12090)), ('an', np.int64(7692)), ('ai', np.int64(6329)), ('developed', np.int64(64878)), ('to', np.int64(166860)), ('provide', np.int64(144583)), ('useful', np.int64(172065)), ('information', np.int64(122645)), ('to', np.int64(166860)), ('users', np.int64(172185)), ('i', np.int64(119007)), ('m', np.int64(129098)), ('conducting', np.int64(42086)), ('a', np.int64(1204)), ('study', np.int64(156078)), ('about', np.int64(4608)), ('data', np.int64(56863)), ('security', np.int64(151344)), ('and', np.int64(8442)), ('the', np.int64(159449)), ('disposal', np.int64(70260)), ('of', np.int64(135465)), ('electronic', np.int64(79490)), ('devices', np.int64(65447))]


## 8. Regression tests before export
Synthetic invalid inputs below test safeguards only; they are never used for training or scoring.

In [11]:
passed = []
def check(name, condition):
    require(bool(condition), name)
    passed.append(name)

def rejects(name, action):
    try:
        action()
    except ValueError:
        passed.append(name)
    else:
        raise AssertionError('Did not reject: ' + name)

base = pd.DataFrame({'Id': ['a', 'b'], 'text': ['hello', 'world'], 'label': ['benign', 'jailbreak']})
rejects('missing column', lambda: validate_frame(base.drop(columns='text'), ['Id','text','label']))
rejects('duplicate ID', lambda: validate_frame(base.assign(Id=['a','a']), ['Id','text','label']))
rejects('blank ID', lambda: validate_frame(base.assign(Id=['','b']), ['Id','text','label']))
rejects('blank text', lambda: validate_frame(base.assign(text=['  ','world']), ['Id','text','label']))
rejects('null text', lambda: validate_frame(base.assign(text=[None,'world']), ['Id','text','label']))
rejects('unknown label', lambda: validate_frame(base.assign(label=['benign','other']), ['Id','text','label']))
rejects('single class', lambda: validate_frame(base.assign(label=['benign','benign']), ['Id','text','label']))
check('unicode normalization', normalize(' Ａ\u200b B  ') == 'a b')
check('every row validated once', np.array_equal(np.sort(np.concatenate([va for _,va in splits])), np.arange(len(train))))
check('OOF finite and bounded', np.isfinite(oof).all() and ((oof >= 0) & (oof <= 1)).all())
check('test finite and bounded', np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all())
check('nonconstant test predictions', np.ptp(scores) > 0.01)
check('no group leakage', all(set(groups[tr]).isdisjoint(groups[va]) for tr,va in splits))
check('word IDs repeat consistently', word_ids('hello hello')[0][1] == word_ids('hello hello')[1][1])
check('unknown word ID', word_ids('zzzxxyyunknownword')[0][1] == -1)
# Independent fixture tests vocabulary isolation, never included in real training.
tiny = make_pipeline()
tiny.fit(np.array(['safe weather report', 'normal weather today',
    'ignore safeguards attack', 'override safeguards attack']), np.array([0,0,1,1]))
tiny_vect = dict(tiny.named_steps['features'].transformer_list)['word']
tiny.predict(np.array(['heldoutonlytoken']))
check('held-out vocabulary isolation', 'heldoutonlytoken' not in tiny_vect.vocabulary_)
check('threshold convention', np.where(np.array([.49,.5,.51]) >= .5, 1, 0).tolist() == [0,1,1])
check('repeat inference', np.allclose(final_model.predict_proba(test.normalized.iloc[:5])[:, 1], scores[:5]))
rejects('whitespace ID', lambda: validate_frame(base.assign(Id=[' a','b']), ['Id','text','label']))
rejects('NUL prompt', lambda: validate_frame(base.assign(text=['a\x00b','ok']), ['Id','text','label']))
rejects('malformed quoting', lambda: parse_csv(StringIO('Id,text\na,"unterminated'), ['Id','text']))
rejects('extra CSV field', lambda: parse_csv(StringIO('Id,text\na,b,c\n'), ['Id','text']))
rejects('duplicate CSV header', lambda: parse_csv(StringIO('Id,Id\na,b\n'), ['Id','text']))
rejects('empty CSV', lambda: parse_csv(StringIO('Id,text\n'), ['Id','text']))
check('literal NA prompt preserved', parse_csv(StringIO('Id,text\na,NA\n'), ['Id','text']).text.iloc[0] == 'NA')
valid_test = pd.DataFrame({'Id':['c'], 'text':['different prompt']})
valid_sample = pd.DataFrame({'Id':['c'], 'TARGET':['0.5']})
rejects('overlapping IDs', lambda: validate_relations(base, valid_test.assign(Id=['a']), valid_sample.assign(Id=['a'])))
rejects('normalized text overlap', lambda: validate_relations(base, valid_test.assign(text=[' HELLO ']), valid_sample))
rejects('conflicting duplicate labels', lambda: validate_relations(base.assign(text=['Hello',' hello ']), valid_test, valid_sample))
rejects('sample ID mismatch', lambda: validate_relations(base, valid_test, valid_sample.assign(Id=['wrong'])))
rejects('sample nonfinite score', lambda: validate_relations(base, valid_test, valid_sample.assign(TARGET=['nan'])))
rejects('sample score out of bounds', lambda: validate_relations(base, valid_test, valid_sample.assign(TARGET=['1.1'])))
check('normalization idempotent', all(normalize(normalize(t)) == normalize(t) for t in train.text.iloc[:100]))
check('inner calibration group isolation', all(set(groups[a]).isdisjoint(groups[b]) for a,b in
    StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(texts,y,groups)))
check('inputs unchanged', all(hashlib.sha256((ROOT/f).read_bytes()).hexdigest() == h for f,h in INPUT_HASHES.items()))
print(f'PASS: {len(passed)} checks before export')
def validate_submission(frame, expected_ids):
    require(list(frame.columns) == ['Id', 'TARGET'], 'Invalid submission schema')
    require(len(frame) == len(expected_ids), 'Submission row count mismatch')
    require(frame.Id.tolist() == list(expected_ids), 'Submission ID order mismatch')
    require(frame.Id.is_unique, 'Duplicate submission IDs')
    require(pd.api.types.is_numeric_dtype(frame.TARGET) and not pd.api.types.is_bool_dtype(frame.TARGET),
            'Submission scores must be numeric, not boolean')
    values = frame.TARGET.to_numpy(dtype=float)
    require(np.isfinite(values).all(), 'Nonfinite submission scores')
    require(((values >= 0) & (values <= 1)).all(), 'Submission scores out of bounds')

fixture = pd.DataFrame({'Id':['x','y'], 'TARGET':[.2,.8]})
rejects('export reordered IDs', lambda: validate_submission(fixture.iloc[::-1], ['x','y']))
rejects('export missing row', lambda: validate_submission(fixture.iloc[:1], ['x','y']))
rejects('export extra column', lambda: validate_submission(fixture.assign(label='benign'), ['x','y']))
rejects('export nonnumeric score', lambda: validate_submission(fixture.assign(TARGET=['.2','bad']), ['x','y']))
rejects('export infinite score', lambda: validate_submission(fixture.assign(TARGET=[np.inf,.8]), ['x','y']))
rejects('export missing score', lambda: validate_submission(fixture.assign(TARGET=[np.nan,.8]), ['x','y']))
rejects('export negative score', lambda: validate_submission(fixture.assign(TARGET=[-.01,.8]), ['x','y']))
rejects('export score above one', lambda: validate_submission(fixture.assign(TARGET=[.2,1.01]), ['x','y']))
rejects('Unicode-equivalent duplicate IDs', lambda: validate_frame(base.assign(Id=['Ａ','A']), ['Id','text','label']))
rejects('invisible format ID', lambda: validate_frame(base.assign(Id=['a\u200bb','c']), ['Id','text','label']))
rejects('unpaired surrogate text', lambda: validate_frame(base.assign(text=['bad\ud800','ok']), ['Id','text','label']))
check('test row permutation invariance', np.allclose(
    final_model.predict_proba(test.normalized.iloc[:5].iloc[::-1])[:,1], scores[:5][::-1], atol=1e-12))
print(f'PASS: {len(passed)} checks before export including adversarial submission fixtures')

latent_estimator = final_pipeline.named_steps['svm']
check('latent basis finite', np.isfinite(latent_estimator.projection_).all())
check('latent rank', latent_estimator.projection_.shape == (len(train), 256))
check('latent training rows only', latent_estimator.training_features_.shape[0] == len(train))
# A row's projection does not depend on other rows being scored.
check('single versus batch scoring', np.allclose(
    final_model.predict_proba(test.normalized.iloc[:1])[:,1], scores[:1], rtol=0, atol=1e-12))
print(f'PASS: {len(passed)} checks before CSV roundtrip')

def export_submission(frame, expected_ids, output, input_hashes):
    """Validate before writing; failures leave an existing submission untouched."""
    validate_submission(frame, expected_ids)
    output = Path(output)
    require(not output.is_symlink(), 'Refusing symlinked output')
    require(output.resolve() not in [Path(f).resolve() for f in input_hashes], 'Output aliases input')
    require(not output.exists() or all(not os.path.samefile(output, Path(f)) for f in input_hashes),
            'Output hard-links an input')
    def verify_sources():
        require(all(hashlib.sha256(Path(f).read_bytes()).hexdigest() == h
                    for f,h in input_hashes.items()), 'Input contents changed during run')
    verify_sources()
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='',
                dir=output.parent, prefix='.submission-', suffix='.tmp', delete=False) as stream:
            temporary = Path(stream.name)
            frame.to_csv(stream, index=False, float_format='%.12g')
            stream.flush()
            os.fsync(stream.fileno())
        restored = pd.read_csv(temporary, dtype={'Id':str}, keep_default_na=False)
        validate_submission(restored, expected_ids)
        require(np.allclose(restored.TARGET, frame.TARGET, rtol=0, atol=1e-11), 'CSV score roundtrip failed')
        verify_sources()
        temporary.replace(output)
        return restored
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)

rejects('boolean TARGET rejected', lambda: validate_submission(fixture.assign(TARGET=[True,False]), ['x','y']))
rejects('nontext prompt control rejected', lambda: validate_frame(base.assign(text=['a\x07b','ok']), ['Id','text','label']))
# Isolated export fixtures are deleted immediately; no additional project files remain.
from unittest.mock import patch
with tempfile.TemporaryDirectory() as directory:
    directory = Path(directory)
    source = directory/'source.csv'
    source.write_text('unchanged source')
    hashes = {source:hashlib.sha256(source.read_bytes()).hexdigest()}
    dest = directory/'submission.csv'
    sentinel = b'previous successful submission\n'
    dest.write_bytes(sentinel)
    rejects('invalid export rejected', lambda: export_submission(fixture.assign(TARGET=[-1,.5]), ['x','y'], dest, hashes))
    check('invalid export preserves existing file', dest.read_bytes() == sentinel)
    rejects('source overwrite rejected', lambda: export_submission(fixture, ['x','y'], source, hashes))
    link = directory/'linked.csv'
    link.symlink_to(source)
    rejects('symlink export rejected', lambda: export_submission(fixture, ['x','y'], link, hashes))
    source.write_text('changed source')
    rejects('changed source aborts export', lambda: export_submission(fixture, ['x','y'], dest, hashes))
    check('source failure preserves existing file', dest.read_bytes() == sentinel)
    source.write_text('unchanged source')
    try:
        with patch.object(pd.DataFrame, 'to_csv', side_effect=OSError('simulated disk error')):
            export_submission(fixture, ['x','y'], dest, hashes)
    except OSError:
        passed.append('disk write failure propagated')
    else:
        raise AssertionError('Simulated disk failure was not propagated')
    check('write failure preserves existing file', dest.read_bytes() == sentinel)
    check('write failure removes temporary files', not list(directory.glob('.submission-*.tmp')))
    # Identifiers that pandas normally interprets as missing must round-trip literally.
    reserved = fixture.assign(Id=['NA','NULL'])
    restored = export_submission(reserved, ['NA','NULL'], dest, hashes)
    check('NA-like IDs preserved by real exporter', restored.Id.tolist() == ['NA','NULL'])
    check('successful export removes temporary files', not list(directory.glob('.submission-*.tmp')))

word_idf_before = word_vect.idf_.copy()
projection_before = latent_estimator.projection_.copy()
_ = final_model.predict_proba(np.array(['zzzxxyyunseenword', 'safe weather request']))
check('inference leaves IDF unchanged', np.array_equal(word_vect.idf_, word_idf_before))
check('inference leaves latent basis unchanged', np.array_equal(latent_estimator.projection_, projection_before))
print(f'PASS: {len(passed)} checks before production export')

# Snapshot integrity is tested with in-memory bytes, not by changing real inputs.
snapshot = b'Id,text\na,hello\n'
snapshot_hash = hashlib.sha256(snapshot).hexdigest()
check('verified snapshot accepted', parse_verified_snapshot(snapshot, snapshot_hash, ['Id','text']).Id.tolist() == ['a'])
rejects('changed snapshot rejected', lambda: parse_verified_snapshot(snapshot+b' ', snapshot_hash, ['Id','text']))
bad_utf8 = b'Id,text\na,\xff\n'
rejects('invalid UTF8 rejected', lambda: parse_verified_snapshot(bad_utf8, hashlib.sha256(bad_utf8).hexdigest(), ['Id','text']))
check('kernel budget accepts supplied data', check_kernel_budget(len(train), len(test)) < MAX_KERNEL_WORKING_BYTES)
rejects('excessive kernel allocation rejected', lambda: check_kernel_budget(100000))
rejects('invalid kernel dimensions rejected', lambda: check_kernel_budget(-1))
rejects('invalid latent weight rejected', lambda: LexicalLatentSVM(latent_weight=2).fit(csr_matrix(np.eye(4)), np.array([0,0,1,1])))
rejects('invalid latent rank rejected', lambda: LexicalLatentSVM(latent_components=0).fit(csr_matrix(np.eye(4)), np.array([0,0,1,1])))
print(f'PASS: {len(passed)} checks before export, including snapshot and allocation guards')



rejects('numeric input ID rejected', lambda: validate_frame(base.assign(Id=[1,2]), ['Id','text','label']))
rejects('surrogate input ID rejected', lambda: validate_frame(base.assign(Id=['bad\ud800','ok']), ['Id','text','label']))
rejects('probability row count rejected', lambda: validate_probability_matrix(np.array([[.2,.8]]), 2))
rejects('probability single column rejected', lambda: validate_probability_matrix(np.array([[.8]]), 1))
rejects('negative complementary probability rejected', lambda: validate_probability_matrix(np.array([[-.1,.8]]), 1))
rejects('nonfinite complementary probability rejected', lambda: validate_probability_matrix(np.array([[np.nan,.8]]), 1))
rejects('non-normalized probability pair rejected', lambda: validate_probability_matrix(np.array([[.1,.8]]), 1))
check('valid binary distribution accepted', validate_probability_matrix(np.array([[.2,.8]]), 1).tolist() == [.8])
with tempfile.TemporaryDirectory() as directory:
    directory = Path(directory)
    source = directory/'source.csv'
    source.write_bytes(b'original input')
    source_bytes = source.read_bytes()
    hashes = {source:hashlib.sha256(source_bytes).hexdigest()}
    alias = directory/'hardlinked.csv'
    os.link(source, alias)
    rejects('hard-linked input export rejected', lambda: export_submission(fixture, ['x','y'], alias, hashes))
    check('hard-linked source preserved', source.read_bytes() == source_bytes)
    destination = directory/'submission.csv'
    sentinel = b'previous successful submission\n'
    destination.write_bytes(sentinel)
    for stage, target, attribute in [
        ('fsync', os, 'fsync'), ('replace', Path, 'replace')]:
        try:
            with patch.object(target, attribute, side_effect=OSError('simulated '+stage+' error')):
                export_submission(fixture, ['x','y'], destination, hashes)
        except OSError:
            passed.append(stage+' failure propagated')
        else:
            raise AssertionError(stage+' failure was not propagated')
        check(stage+' failure preserves prior submission', destination.read_bytes() == sentinel)
        check(stage+' failure cleans temporary file', not list(directory.glob('.submission-*.tmp')))
    with patch.object(pd, 'read_csv', return_value=fixture.assign(Id=['bad','ids'])):
        rejects('corrupt readback aborts export', lambda: export_submission(fixture, ['x','y'], destination, hashes))
    check('corrupt readback preserves prior submission', destination.read_bytes() == sentinel)
    check('corrupt readback cleans temporary file', not list(directory.glob('.submission-*.tmp')))
print(f'PASS: {len(passed)} checks before production export')

check('full-data classifier refit', latent_estimator.training_features_.shape[0] == len(train))
check('lexical submodel has only word and character views', [name for name,_ in final_pipeline.named_steps['features'].transformer_list] == ['word','char'])
check('one full-data calibration classifier', len(final_model.calibrated_classifiers_) == 1)
final_distribution = final_model.predict_proba(test.normalized.iloc[:5])
check('complete prediction distribution valid', np.allclose(
    validate_probability_matrix(final_distribution, 5), scores[:5], rtol=0, atol=1e-12))
print(f'PASS: {len(passed)} checks before final production export')

# Pretrained asset integrity and row-local semantic invariants.
small_payload = b'verified fixture'
small_digest = hashlib.sha256(small_payload).hexdigest()
check('valid encoder payload accepted', verify_asset_payload(small_payload,small_digest,len(small_payload)) == small_payload)
rejects('encoder payload size mismatch rejected', lambda: verify_asset_payload(small_payload,small_digest,len(small_payload)+1))
rejects('encoder payload hash mismatch rejected', lambda: verify_asset_payload(small_payload,'0'*64,len(small_payload)))
rejects('invalid semantic blend rejected', lambda: SemanticLexicalClassifier(semantic_weight=2).fit(np.array(['hi']),np.array([1])))
check('semantic model refit on all labels', final_hybrid.semantic_training_rows_ == len(train))
check('semantic model class direction', final_hybrid.semantic_model_.classes_.tolist() == [0,1])
check('encoder cache array is read-only', not frozen_embedding(texts[0]).flags.writeable)
check('embedding unit norm', np.isclose(np.linalg.norm(frozen_embedding(texts[0])),1))
check('embedding dimension', frozen_embedding(texts[0]).shape == (384,))
check('semantic feature row order', np.array_equal(semantic_features(texts[:3][::-1]), semantic_features(texts[:3])[::-1]))
check('semantic canonical normalization', np.array_equal(frozen_embedding(' HELLO  world '), frozen_embedding('hello world')))
long_fixture = 'hello world '*400
frozen_embedding(long_fixture)
check('long prompt uses multiple chunks', _chunk_counts[normalize(long_fixture)] > 1)
check('long prompt tail retained', len(wordpiece.encode(normalize(long_fixture),add_special_tokens=False).ids) > 254)
for name,(_,digest,size) in ASSETS.items():
    check(name+' remains checksum verified', hashlib.sha256((ASSET_DIR/name).read_bytes()).hexdigest() == digest)
print(f'PASS: {len(passed)} checks before production export including frozen encoder safeguards')

changed_tail = long_fixture + ' different tail meaning'
check('semantic embedding includes changed tail', not np.array_equal(
    frozen_embedding(long_fixture), frozen_embedding(changed_tail)))
print(f'PASS: {len(passed)} checks before final CSV roundtrip')

# Fold-local learned semantic statistics must exclude validation/test rows.
semantic_scaler = final_hybrid.semantic_model_.named_steps['scale']
train_embeddings = semantic_features(texts)
check('semantic scaler fits exactly training rows', semantic_scaler.n_samples_seen_ == len(train))
check('semantic scaler mean matches training only', np.allclose(semantic_scaler.mean_,train_embeddings.mean(axis=0),rtol=0,atol=1e-12))
check('semantic scaler variance matches training only', np.allclose(semantic_scaler.var_,train_embeddings.var(axis=0),rtol=0,atol=1e-12))
mean_before = semantic_scaler.mean_.copy()
scale_before = semantic_scaler.scale_.copy()
final_model.predict_proba(test.normalized.iloc[:5])
check('inference leaves semantic mean unchanged', np.array_equal(semantic_scaler.mean_,mean_before))
check('inference leaves semantic scale unchanged', np.array_equal(semantic_scaler.scale_,scale_before))
# A synthetic held-out extreme value must not affect a scaler fitted on training fixtures.
fixture_semantics = np.array([[0.,1.],[1.,0.],[0.,2.],[2.,0.]])
fixture_labels = np.array([0,1,0,1])
fixture_pipeline = make_semantic_pipeline()
fixture_pipeline.fit(fixture_semantics,fixture_labels)
fixture_mean = fixture_pipeline.named_steps['scale'].mean_.copy()
fixture_pipeline.predict(np.array([[1000.,-1000.]]))
check('held-out semantic statistics isolation', np.array_equal(fixture_mean,fixture_semantics.mean(axis=0)) and np.array_equal(fixture_pipeline.named_steps['scale'].mean_,fixture_mean))
rejects('overlong CSV prompt rejected', lambda: validate_frame(base.assign(text=['x'*(MAX_PROMPT_CHARACTERS+1),'ok']), ['Id','text','label']))
rejects('overlong semantic prompt rejected', lambda: frozen_embedding('x'*(MAX_PROMPT_CHARACTERS+1)))
rejects('blank semantic prompt rejected', lambda: frozen_embedding('  '))
rejects('semantic token budget rejected', lambda: check_semantic_token_budget(MAX_SEMANTIC_TOKENS+1))
rejects('negative semantic token count rejected', lambda: check_semantic_token_budget(-1))
rejects('fractional semantic token count rejected', lambda: check_semantic_token_budget(1.5))
check('semantic token budget boundary accepted', check_semantic_token_budget(MAX_SEMANTIC_TOKENS) is None)
rejects('zero encoder vector rejected', lambda: normalize_encoder_vector(np.zeros(384)))
rejects('NaN encoder vector rejected', lambda: normalize_encoder_vector(np.full(384,np.nan)))
rejects('infinite encoder vector rejected', lambda: normalize_encoder_vector(np.full(384,np.inf)))
rejects('wrong encoder dimension rejected', lambda: normalize_encoder_vector(np.ones(383)))
check('finite encoder normalization accepted', np.isclose(np.linalg.norm(normalize_encoder_vector(np.ones(384))),1))
print(f'PASS: {len(passed)} checks before final submission export')


PASS: 34 checks before export
PASS: 46 checks before export including adversarial submission fixtures


PASS: 50 checks before CSV roundtrip
PASS: 65 checks before production export
PASS: 73 checks before export, including snapshot and allocation guards
PASS: 92 checks before production export


PASS: 96 checks before final production export
PASS: 111 checks before production export including frozen encoder safeguards


PASS: 112 checks before final CSV roundtrip


PASS: 130 checks before final submission export


## 9. Atomic submission export and inline audit report
The only exported file is `submission.csv`, exactly `Id,TARGET`, preserving test ID spelling/order. All checks, input hashes, versions, metrics, and limitations are printed below and saved as notebook outputs (no report sidecar).

In [12]:
submission = pd.DataFrame({'Id': test.Id, 'TARGET': scores})
roundtrip = export_submission(submission, test.Id.tolist(), ROOT/'submission.csv',
    {ROOT/f:h for f,h in INPUT_HASHES.items()})
check('CSV schema', roundtrip.columns.tolist() == ['Id','TARGET'])
check('CSV ID order', roundtrip.Id.tolist() == sample.Id.tolist())
check('CSV row count', len(roundtrip) == len(test))
check('CSV probabilities', np.allclose(roundtrip.TARGET, scores, rtol=0, atol=1e-11))
check('CSV probability bounds', roundtrip.TARGET.between(0,1).all())
check('input hashes verified after export', all(
    hashlib.sha256((ROOT/f).read_bytes()).hexdigest() == h for f,h in INPUT_HASHES.items()))
report = {
    'model': '70/30 lexical-latent / training-standardized MiniLM RBF-SVM, grouped sigmoid calibration',
    'selection': 'Measured complementarity of pretrained semantic and lexical views; public target unverified',
    'best_prior_user_reported_public_score': .97482,
    'kernel_working_set_limit_bytes': MAX_KERNEL_WORKING_BYTES,
    'semantic_preprocessing': 'training-only StandardScaler then L2 Normalizer',
    'prompt_character_budget': MAX_PROMPT_CHARACTERS, 'semantic_token_budget': MAX_SEMANTIC_TOKENS,
    'semantic_weight': SEMANTIC_WEIGHT, 'semantic_dimensions': SEMANTIC_DIMENSION,
    'pretrained_asset_repository': ASSET_REPOSITORY, 'pretrained_asset_commit': ASSET_SOURCE_COMMIT,
    'pretrained_asset_hashes': {name:digest for name,(_,digest,_) in ASSETS.items()},
    'latent_components': 256, 'min_df': 1, 'word_ngram_range': [1,3],
    'sensitivity_split_seed': 2026, 'sensitivity_split_roc_auc': sensitivity_auc,
    'seed': SEED, 'folds': FOLDS, 'training_rows': len(train), 'test_rows': len(test),
    'removed_exact_duplicates': duplicate_rows,
    'class_counts': train.label.value_counts().to_dict(), 'near_test_prompts_audit': near_test_count, 'mixed_label_lexical_groups': mixed_groups,
    'near_duplicate_cosine': NEAR_DUPLICATE_COSINE, 'groups': len(group_rows),
    'development_roc_auc': metrics, 'fold_roc_auc': fold_metrics,
    'group_bootstrap_95_interval': auc_interval,
    'accuracy_at_0.5': float(accuracy_score(y, oof_score >= .5)),
    'f1_at_0.5': float(f1_score(y, oof_score >= .5)),
    'brier': float(brier_score_loss(y, oof_score)),
    'average_precision': float(average_precision_score(y, oof_score)),
    'tests_passed': passed,
    'input_sha256': {f: hashlib.sha256((ROOT/f).read_bytes()).hexdigest()
                     for f in ['train.csv','test.csv','sample_submission.csv']},
    'versions': {'python': platform.python_version(), 'numpy': np.__version__,
                 'pandas': pd.__version__, 'scipy': scipy.__version__, 'sklearn': sklearn.__version__, 'onnxruntime':ort.__version__, 'tokenizers':tokenizers.__version__},
    'submission_sha256': hashlib.sha256((ROOT/'submission.csv').read_bytes()).hexdigest(),
    'public_leaderboard_score': None, 'user_reported_previous_public_score': 0.97482,
    'limitations': ['Lexical grouping cannot guarantee semantic or source independence.',
                    'Likelihoods are not independently calibrated.',
                    'Model family and hyperparameters compared on reused development folds.',
                    'Public leaderboard target 0.98 is unverified.',
                    'Pretrained encoder is external data; verify competition rules and model licensing.'],
}
print(json.dumps(report, indent=2))
print(f'PASS: {len(passed)} checks; exported {len(submission)} rows to submission.csv')
print(f'Runtime: {time.monotonic()-START:.1f} seconds')

{
  "model": "70/30 lexical-latent / training-standardized MiniLM RBF-SVM, grouped sigmoid calibration",
  "selection": "Measured complementarity of pretrained semantic and lexical views; public target unverified",
  "best_prior_user_reported_public_score": 0.97482,
  "kernel_working_set_limit_bytes": 2147483648,
  "semantic_preprocessing": "training-only StandardScaler then L2 Normalizer",
  "prompt_character_budget": 100000,
  "semantic_token_budget": 32768,
  "semantic_weight": 0.3,
  "semantic_dimensions": 384,
  "pretrained_asset_repository": "AOSSIE-Org/MoveYourBody",
  "pretrained_asset_commit": "53c128d1d8555d71d65fdd6bc035a27743dd8dd6",
  "pretrained_asset_hashes": {
    "model.onnx": "321dc4b9e00da5e056106d884e02094b678583bee558796eb66f66e4f26ede83",
    "vocab.txt": "07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3"
  },
  "latent_components": 256,
  "min_df": 1,
  "word_ngram_range": [
    1,
    3
  ],
  "sensitivity_split_seed": 2026,
  "sensitivity_split_